In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect('lab3_blockchain.db')
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

In [3]:
cur.executescript("""
CREATE TABLE IF NOT EXISTS BLOCKS (
    hexString TEXT PRIMARY KEY,
    id INTEGER,
    view INTEGER,
    desc TEXT,
    img BLOB,
    votes INTEGER
);

CREATE TABLE IF NOT EXISTS SOURCES (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ip_addr TEXT,
    country_code TEXT
);

CREATE TABLE IF NOT EXISTS PERSONS (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    addr TEXT
);

CREATE TABLE IF NOT EXISTS VOTES (
    block_id TEXT,
    voter_id INTEGER,
    timestamp DATETIME,
    source_id INTEGER,
    PRIMARY KEY (block_id, voter_id),
    FOREIGN KEY (block_id) REFERENCES BLOCKS(hexString) ON DELETE CASCADE,
    FOREIGN KEY (voter_id) REFERENCES PERSONS(id) ON DELETE CASCADE,
    FOREIGN KEY (source_id) REFERENCES SOURCES(id) ON DELETE SET NULL
);
""")

In [4]:
blocks = [
    ('0x1f3a5b7c', 2001, 500, 'First block', None, 1),
    ('0x8e2d4f6a', 2002, 600, 'Second block', None, 2),
    ('0x9c0b3d1e', 2003, 700, 'Third block', None, 3)
]
cur.executemany("INSERT OR IGNORE INTO BLOCKS VALUES (?, ?, ?, ?, ?, ?)", blocks)

persons = [
    (1, 'Іван Петренко', 'Київ, вул. Хрещатик, 10'),
    (2, 'Олена Коваленко', 'Львів, пл. Ринок, 5'),
    (3, 'Михайло Шевченко', 'Одеса, вул. Дерибасівська, 20')
]
cur.executemany("INSERT OR IGNORE INTO PERSONS (id, name, addr) VALUES (?, ?, ?)", persons)

sources = [
    (1, '203.0.113.1', 'UA'),
    (2, '198.51.100.2', 'PL'),
    (3, '192.0.2.3', 'DE'),
    (4, '10.0.0.4', 'US')
]
cur.executemany("INSERT OR IGNORE INTO SOURCES (id, ip_addr, country_code) VALUES (?, ?, ?)", sources)

votes = [
    ('0x1f3a5b7c', 1, '2025-02-10 12:00:00', 1),
    ('0x1f3a5b7c', 2, '2025-02-10 12:05:00', 2),
    ('0x8e2d4f6a', 1, '2025-02-10 12:10:00', 3),
    ('0x8e2d4f6a', 3, '2025-02-10 12:15:00', 1),
    ('0x9c0b3d1e', 2, '2025-02-10 12:20:00', 4),
    ('0x9c0b3d1e', 3, '2025-02-10 12:25:00', 2)
]
cur.executemany("INSERT INTO VOTES (block_id, voter_id, timestamp, source_id) VALUES (?, ?, ?, ?)", votes)

conn.commit()

In [5]:
print("--- BLOCKS ---")
for row in cur.execute("SELECT hexString, id, view, desc, votes FROM BLOCKS"):
    print(row)

print("\n--- PERSONS ---")
for row in cur.execute("SELECT id, name, addr FROM PERSONS"):
    print(row)

print("\n--- SOURCES ---")
for row in cur.execute("SELECT id, ip_addr, country_code FROM SOURCES"):
    print(row)

print("\n--- VOTES (з деталями) ---")
cur.execute("""
    SELECT v.block_id, v.voter_id, v.timestamp, v.source_id,
           p.name, s.ip_addr, s.country_code, b.desc
    FROM VOTES v
    LEFT JOIN PERSONS p ON v.voter_id = p.id
    LEFT JOIN SOURCES s ON v.source_id = s.id
    LEFT JOIN BLOCKS b ON v.block_id = b.hexString
    ORDER BY v.timestamp
""")
for row in cur.fetchall():
    print(f"{row[0]} ({row[7]}), voter {row[1]} ({row[4]}), time {row[2]}, source {row[3]} ({row[5]}, {row[6]})")

--- BLOCKS ---
('0x1f3a5b7c', 2001, 500, 'First block', 1)
('0x8e2d4f6a', 2002, 600, 'Second block', 2)
('0x9c0b3d1e', 2003, 700, 'Third block', 3)

--- PERSONS ---
(1, 'Іван Петренко', 'Київ, вул. Хрещатик, 10')
(2, 'Олена Коваленко', 'Львів, пл. Ринок, 5')
(3, 'Михайло Шевченко', 'Одеса, вул. Дерибасівська, 20')

--- SOURCES ---
(1, '203.0.113.1', 'UA')
(2, '198.51.100.2', 'PL')
(3, '192.0.2.3', 'DE')
(4, '10.0.0.4', 'US')

--- VOTES (з деталями) ---
0x1f3a5b7c (First block), voter 1 (Іван Петренко), time 2025-02-10 12:00:00, source 1 (203.0.113.1, UA)
0x1f3a5b7c (First block), voter 2 (Олена Коваленко), time 2025-02-10 12:05:00, source 2 (198.51.100.2, PL)
0x8e2d4f6a (Second block), voter 1 (Іван Петренко), time 2025-02-10 12:10:00, source 3 (192.0.2.3, DE)
0x8e2d4f6a (Second block), voter 3 (Михайло Шевченко), time 2025-02-10 12:15:00, source 1 (203.0.113.1, UA)
0x9c0b3d1e (Third block), voter 2 (Олена Коваленко), time 2025-02-10 12:20:00, source 4 (10.0.0.4, US)
0x9c0b3d1e (Third 

In [6]:
cur.close()
conn.close()
print("З'єднання закрито.")

З'єднання закрито.
